# Kürzeste Wege – Von Dijkstra zu A*

## Szenario: Routenplanung zwischen DHBW-Standorten

In diesem Notebook untersuchen wir zwei grundlegende Suchverfahren zur Bestimmung kürzester Wege in gewichteten Graphen:

- den **Dijkstra-Algorithmus** als uninformierte Suche,
- und **A\*** als informierte Suche mit Heuristik.

Als Beispiel verwenden wir ein vereinfachtes Netz von **DHBW-Standorten**. Die Kanten des Graphen repräsentieren Verbindungen zwischen den Standorten und sind mit Entfernungen gewichtet.

Ziel ist es, den kürzesten Weg von **Lörrach (LÖ)** nach **Heidenheim (HDH)** zu finden.

> **Wichtig:** Dijkstra und A* können denselben optimalen Weg finden. Der entscheidende Unterschied liegt darin, **welche Knoten während der Suche untersucht werden und in welcher Reihenfolge dies geschieht**.

## Lernziele

1. einen **gewichteten Graphen** als Adjazenzstruktur in Python darstellen,

2. die grundlegende Funktionsweise des **Dijkstra-Algorithmus** erklären

3. zwischen **uninformierter und informierter Suche** unterscheiden,

4. erklären, welche Rolle eine **Heuristik** bei A* spielt

5. eine Dijkstra-Suche zu einer einfachen **A\*-Suche** erweitern,

6. Dijkstra und A* anhand der **Anzahl expandierter Knoten** vergleichen.

## Ablauf

Das Notebook gliedert sich in vier Schritte:

**1. Dijkstra verstehen**  
Machen Sie sich mit der gegebenen Implementierung, der Priority Queue und der Berechnung der kürzesten Distanzen vertraut.

**2. Dijkstra auf ein Routennetz anwenden**  
Berechnen Sie den kürzesten Weg von Lörrach nach Heidenheim und untersuchen Sie Distanz, Vorgängerknoten und Anzahl der besuchten Knoten.

**3. Eine Heuristik ergänzen**  
Berechnen Sie für die Standorte die Luftliniendistanzen und verwenden Sie die Entfernung zum Ziel als zusätzliche Information.

**4. Dijkstra und A*** **vergleichen**  
Untersuchen Sie, wie sich die Heuristik auf die Reihenfolge der Suche und insbesondere auf die Anzahl der expandierten Knoten auswirkt.

Die zentrale Frage des Notebooks lautet:

> **Wie kann zusätzliche Information über die Entfernung zum Ziel eine Suche effizienter machen, ohne den kürzesten Weg aus den Augen zu verlieren?**

# 1. Dijkstra verstehen

Machen Sie sich mit folgender Implementierung von Dijkstra für gerichtete Graphen vertraut:

In [1]:
from heapq import heapify, heappop, heappush
import itertools

from networkx import is_empty


class PriorityQueue:
    sentinel = object()  # create a unique empty object that never conflicts with real items & can be checked for equality fast (using "is")
    count = 0

    def __init__(self):
        # entries stores a triple: priority, a unique increasing tie-breaker 
        #                          if equal priorities, and a node (item)
        self.heap, self.entries = [], {}
        self.counter = itertools.count()

    def push(self, item, priority):
        if item in self.entries:
            self.entries[item][2] = self.sentinel # mark old entry as invalid

        entry = self.entries[item] = [priority, next(self.counter), item] # update item
        heappush(self.heap, entry)
        self.count = self.count + 1

    def pop(self):
        while True:
            priority, count, item = heappop(self.heap)
            self.count = self.count - 1

            if item is not self.sentinel:
                del self.entries[item]
                return item


class Graph: 
   def __init__(self, graph: dict = {}):
       self.graph = graph  # A dictionary for the adjacency list

   def add_edge(self, node1, node2, weight):
       if node1 not in self.graph:  # Check if the node is already added
           self.graph[node1] = {}  # If not, create the node
       self.graph[node1][node2] = weight  # Else, add a connection to its neighbor


sentinel2 = object()

def shortest_distances(graph: Graph, source: str, target = sentinel2):
    
   # Initialize the values of all nodes with infinity
   dist = {node: float("inf") for node in graph.graph}
   dist[source] = 0  # Set the source value to 0
   # Initialize array of predecessor nodes
   prev = {node: "-" for node in graph.graph}

   # Initialize a priority queue Q
   Q = PriorityQueue()
   Q.push(source, 0)

   abort = False # used to prematurely end iterations if target node given and reached
   nodeCount = 0
   while Q.entries:  # While the priority queue isn't empty
       u = Q.pop() # Get the node with the min distance
       nodeCount += 1 # count the number of visited nodes
       # Go over all neighboring nodes of current node u
       for v, weight in graph.graph[u].items():
              # Calculate the distance from current node u to its neighbor v
              if dist[v] == float("inf"): # never visited before
                  dist[v] = dist[u] + weight
                  prev[v] = u
                  Q.push(v, dist[v])
              elif dist[u] + weight < dist[v]: # visited before
                  dist[v] = dist[u] + weight
                  prev[v] = u
                  Q.push(v, dist[v]) # push is implemented such that it does an update
              # if we have visited the target node, we can abort
              if target is not sentinel2 and u == target:
                  print("target node reached, abort")
                  abort = True
                  break;

       if abort:
           break
   return dist, prev, nodeCount


# 2. Dijkstra auf ein Routennetz anwenden

Verwenden Sie nun den untenstehenden Code, um den Graphen der DHBW Standorte (siehe Skript) zu generieren und den kürzesten Weg von Lörrach (LÖ) nach Heidenheim (HDH) zu berechnen. Geben Sie die kürzeste Distanz, die Vorgängerknoten und die Anzahl der expandierten Knoten aus:

In [2]:
# Create graph, since the Dijkstra implementation is for directed graphs but the distance graph is undirected, we add each edge twice
G = Graph()
G.add_edge('LÖ', 'VS', 110)
G.add_edge('VS', 'LÖ', 110)
G.add_edge('VS', 'RV', 110)
G.add_edge('RV', 'VS', 110)
G.add_edge('VS', 'S', 120)
G.add_edge('S', 'VS', 120)
G.add_edge('RV', 'HDH', 130)
G.add_edge('HDH', 'RV', 130)
G.add_edge('VS', 'KA', 150)
G.add_edge('KA', 'VS', 150)
G.add_edge('KA', 'MN', 70)
G.add_edge('MN', 'KA', 70)
G.add_edge('KA', 'S', 75)
G.add_edge('S', 'KA', 75)
G.add_edge('S', 'HN', 55)
G.add_edge('HN', 'S', 55)
G.add_edge('S', 'HDH', 85)
G.add_edge('HDH', 'S', 85)
G.add_edge('S', 'HN', 55)
G.add_edge('HN', 'S', 55)
G.add_edge('HN', 'MN', 80)
G.add_edge('MN', 'HN', 80)
G.add_edge('HN', 'MOS', 30)
G.add_edge('MOS', 'HN', 30)
G.add_edge('HN', 'HDH', 130)
G.add_edge('HDH', 'HN', 130)

# compute shortest distance(s)
# TODO
dist, prev, nodeCount = shortest_distances(G, 'LÖ', 'HDH')
print("distances:", dist)
print("predecessors:", prev)
print("number of visited nodes:", nodeCount)


target node reached, abort
distances: {'LÖ': 0, 'VS': 110, 'RV': 220, 'S': 230, 'HDH': 315, 'KA': 260, 'MN': 330, 'HN': 285, 'MOS': 315}
predecessors: {'LÖ': '-', 'VS': 'LÖ', 'RV': 'VS', 'S': 'VS', 'HDH': 'S', 'KA': 'VS', 'MN': 'KA', 'HN': 'S', 'MOS': 'HN'}
number of visited nodes: 7


Folgender Code generiert eine Distanz-Matrix dist_matrix, die die Luftlinie zwischen jeweils zwei der Städte enthält.

In [3]:
import numpy as np
import math

# list + coords
cities = ["LÖ","S","HDH","HN","MN","MOS","KA","VS","RV"]
coords = {
    "LÖ": (47.6169, 7.6709),
    "S":  (48.7823, 9.1770),
    "HDH":(48.6780, 10.1516),
    "HN": (49.1427, 9.2109),
    "MN": (49.4891, 8.4669),
    "MOS":(49.3489, 9.1294),
    "KA": (49.0094, 8.4044),
    "VS": (48.0594, 8.4641),
    "RV": (47.7820,9.6167),
}

# create dictionary, that to each city assigns its index in the list, e.g. {'HDH': 0, 'HN': 1, ...}
city_index = {c: i for i, c in enumerate(cities)} 

# Haversine formula
def haversine(a, b):
    R = 6371.0  # Earth radius in km
    lat1, lon1 = map(math.radians, a)
    lat2, lon2 = map(math.radians, b)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    u = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return 2*R*math.atan2(math.sqrt(u), math.sqrt(1-u))

# Build symmetric matrix
n = len(cities)
dist_matrix = np.zeros((n, n), float)
for i in range(n):
    for j in range(i+1, n):
        d = haversine(coords[cities[i]], coords[cities[j]])
        dist_matrix[i, j] = d
        dist_matrix[j, i] = d



Daraus erzeugen wir eine Methode get_direct_distance(a, b), die die Luftlinie der Städte a und b zurückliefert. Testen Sie die Funktion. Die Distanz von LÖ nach HDH ist z.B. 218.6 km.

In [4]:
# Lookup function
def get_direct_distance(a, b):
    return dist_matrix[city_index[a], city_index[b]]

# TODO
get_direct_distance("LÖ", "HDH")

np.float64(218.60163812672434)

# 3. Eine Heuristik ergänzen

Machen Sie aus obiger shortest_distances Suchfunktion eine informierte Suche. Ändern Sie dazu die Methode so ab, dass eine Heuristik verwendet wird. Dazu verwenden Sie bitte die Luftlinie, die durch obige Methode bereitgestellt wird. 

In [5]:
# TODO

def shortest_distances_heuristic(graph: Graph, source: str, target = sentinel2):
    
   # Initialize the values of all nodes with infinity
   dist = {node: float("inf") for node in graph.graph}
   dist[source] = 0  # Set the source value to 0
   # Initialize array of predecessor nodes
   prev = {node: "-" for node in graph.graph}

   # Initialize a priority queue Q
   Q = PriorityQueue()
   Q.push(source, 0)

   abort = False # used to prematurely end iterations if target node given and reached
   nodeCount = 0
   while Q.entries:  # While the priority queue isn't empty
       u = Q.pop() # Get the node with the min distance
       nodeCount += 1 # count the number of visited nodes
       # Go over all neighboring nodes of current node u
       for v, weight in graph.graph[u].items():
              # Calculate the distance from current node u to its neighbor v
              if dist[v] == float("inf"): # never visited before
                  dist[v] = dist[u] + weight
                  prev[v] = u
                  Q.push(v, dist[v] + get_direct_distance(v, target) if target is not sentinel2 else 0)
              elif dist[u] + weight < dist[v]: # visited before
                  dist[v] = dist[u] + weight
                  prev[v] = u
                  Q.push(v, dist[v] + get_direct_distance(v, target) if target is not sentinel2 else 0)
              # if we have visited the target node, we can abort
              if target is not sentinel2 and u == target:
                  print("target node reached, abort")
                  abort = True
                  break;

       if abort:
           break
   return dist, prev, nodeCount



# 4. Dijkstra und A* vergleichen

Anschließend testen Sie die Methode erneut:

In [6]:

# compute shortest distance(s) with PriorityQueue and A* heuristic

dist, prev, nodeCount = shortest_distances(G, 'LÖ', 'HDH')
print("distances:", dist)
print("predecessors:", prev)
print("number of visited nodes:", nodeCount)

dist, prev, nodeCount = shortest_distances_heuristic(G, 'LÖ', 'HDH')
print("distances:", dist)
print("predecessors:", prev)
print("number of visited nodes:", nodeCount)


target node reached, abort
distances: {'LÖ': 0, 'VS': 110, 'RV': 220, 'S': 230, 'HDH': 315, 'KA': 260, 'MN': 330, 'HN': 285, 'MOS': 315}
predecessors: {'LÖ': '-', 'VS': 'LÖ', 'RV': 'VS', 'S': 'VS', 'HDH': 'S', 'KA': 'VS', 'MN': 'KA', 'HN': 'S', 'MOS': 'HN'}
number of visited nodes: 7
target node reached, abort
distances: {'LÖ': 0, 'VS': 110, 'RV': 220, 'S': 230, 'HDH': 315, 'KA': 260, 'MN': inf, 'HN': 285, 'MOS': inf}
predecessors: {'LÖ': '-', 'VS': 'LÖ', 'RV': 'VS', 'S': 'VS', 'HDH': 'S', 'KA': 'VS', 'MN': '-', 'HN': 'S', 'MOS': '-'}
number of visited nodes: 4


Was fällt auf? Insbesondere hinsichtlich der Anzahl der expandierten Knoten? 

Es wurden weniger Knoten Expandiert als bei der Suche ohne Heuristik

*Quellenhinweis*: Bei der Formattierung der Einleitung dieses Notebooks wurde ChatGPT-5.1 verwendet.